# Tutorial: Delegated Authentication Demo

Audience:
- Power BI and Fabric presenters
- Analysts and BI developers who want a notebook-first demo

Prerequisites:
- `.env` filled in locally
- manual prerequisites completed in `docs/setup_checklist.md`
- signed-in user has workspace access and dataset Build for execute queries

Learning goals:
- Load local configuration safely
- Acquire a delegated Power BI token with device code flow
- List workspaces, datasets, and reports
- Run one small DAX query and display the result in pandas


## Outline

1. Load config
2. Acquire a delegated token
3. List workspaces
4. List datasets and reports
5. Execute a small DAX query

This notebook uses device code flow by default because it is the least fragile local demo path.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.auth.delegated_auth import acquire_delegated_access_token
from src.clients.powerbi_client import PowerBIClient
from src.utils.config import load_config, require_dataset_id, require_group_id

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## Step 1 - Load config

This reads `.env`, validates the required values, and shows a redacted summary so secrets are never printed.

In [ ]:
config = load_config()
config.redacted_summary()


## Step 2 - Acquire a delegated token

By default, this uses device code flow. If you later configure a localhost redirect URI and want browser sign-in, call `acquire_delegated_access_token(config, use_device_code=False)` instead.

In [ ]:
access_token = acquire_delegated_access_token(config, use_device_code=True)
client = PowerBIClient(access_token, timeout_seconds=config.request_timeout_seconds)
print("Delegated access token acquired.")


## Step 3 - List workspaces

This shows the workspaces visible to the signed-in user. If your target workspace is missing, the problem is almost always user access or tenant policy, not the notebook code.

In [ ]:
workspaces_df = pd.DataFrame(client.get_groups())
workspaces_df


## Step 4 - Select the workspace

The notebook uses `WORKSPACE_ID` from `.env`. You can override it manually in the next cell if you want to pick a different workspace from the table above.

In [ ]:
group_id = require_group_id(None, config)
group_id


## Step 5 - List datasets and reports

These calls depend on workspace access. If datasets list correctly but query execution fails later, check dataset Build permission first.

In [ ]:
datasets_df = pd.DataFrame(client.get_datasets_in_group(group_id))
reports_df = pd.DataFrame(client.get_reports_in_group(group_id))

print("Datasets")
display(datasets_df)
print("Reports")
display(reports_df)


## Step 6 - Execute a very small DAX query

This step depends on dataset Read and Build permissions. Keep the query short and presentation-friendly.

In [ ]:
dataset_id = require_dataset_id(None, config)
query_df = pd.DataFrame(
    client.execute_queries_in_group(
        group_id=group_id,
        dataset_id=dataset_id,
        dax_query=config.dax_query,
        impersonated_user_name=config.impersonated_user_name,
    )
)
query_df


## Presenter notes

- This is the recommended demo path.
- Device code flow avoids redirect URI setup friction.
- The signed-in user sees only what they can really access.
- If `executeQueries` fails, mention dataset Build permission and tenant settings before changing code.
